# Step 02 — Build the HMM
**NLP Course Project · Statistical POS Tagging using HMMs**  
23BCS052 · Hammad Malik

---

### What this notebook builds
The three core components of an HMM tagger:

| Component | Symbol | What it answers |
|---|---|---|
| Transition Matrix | **A** | How likely is tag *j* to follow tag *i*? |
| Emission Matrix | **B** | How likely is word *w* given tag *t*? |
| Initial Probabilities | **π** | How likely is tag *t* to start a sentence? |

> 🧠 **Mental model:** Think of A as *grammar patterns* (Determiner → Noun is common),  
> B as *word personalities* (the word 'runs' almost always acts as a Verb),  
> and π as *sentence openers* (sentences usually start with a Noun or Pronoun).

---
## Cell 1 — Imports & Load Processed Data

In [ ]:
import pickle
import numpy as np
from collections import defaultdict, Counter

# Load everything we saved in Step 01
with open('/content/pos_data.pkl', 'rb') as f:
    data = pickle.load(f)

# ── Check what keys actually exist ──
print('Keys found in pkl:', list(data.keys()))
print()

train_sents = data['train_sents']
test_sents  = data['test_sents']

# ── Rebuild vocab, tagset and lookups from the raw sentences ──
# (These are quick to recompute from the training sentences)

train_words = [word for sent in train_sents for word, tag in sent]
train_tags  = [tag  for sent in train_sents for word, tag in sent]

vocab   = sorted(set(train_words)) + ['<UNK>']
tagset  = sorted(set(train_tags))

word2idx = {w: i for i, w in enumerate(vocab)}
tag2idx  = {t: i for i, t in enumerate(tagset)}
idx2tag  = {i: t for t, i in tag2idx.items()}

T = len(tagset)
V = len(vocab)

print('✅ Data loaded & lookups rebuilt successfully')
print(f'   Tags in tagset  : {T}')
print(f'   Words in vocab  : {V:,}')
print(f'   Train sentences : {len(train_sents):,}')
print(f'   Test  sentences : {len(test_sents):,}')

Keys found in pkl: ['train_sents', 'test_sents', 'train_vocab', 'tag_counts', 'all_tags']

✅ Data loaded & lookups rebuilt successfully
   Tags in tagset  : 44
   Words in vocab  : 10,109
   Train sentences : 3,131
   Test  sentences : 783


---
## Cell 2 — Count Everything

> 🧠 Before probabilities come **counts**. We make three tallies:
> - `transition_counts[i][j]` — how many times tag *j* followed tag *i*
> - `emission_counts[i][k]`   — how many times word *k* appeared with tag *i*
> - `init_counts[i]`          — how many sentences started with tag *i*
>
> We also track `tag_totals[i]` — total times tag *i* appeared — so we can normalise.

In [ ]:
# Initialise count arrays with zeros
# transition_counts : shape (T, T)  — tag-to-tag
# emission_counts   : shape (T, V)  — tag-to-word
# init_counts       : shape (T,)    — sentence-start tags
# tag_totals        : shape (T,)    — total occurrences of each tag

transition_counts = np.zeros((T, T), dtype=np.float64)
emission_counts   = np.zeros((T, V), dtype=np.float64)
init_counts       = np.zeros(T,      dtype=np.float64)
tag_totals        = np.zeros(T,      dtype=np.float64)

# ── Single pass through all training sentences ──
for sent in train_sents:
    if len(sent) == 0:
        continue

    # First word of the sentence → initial probability
    first_word, first_tag = sent[0]
    if first_tag in tag2idx:
        init_counts[tag2idx[first_tag]] += 1

    for idx, (word, tag) in enumerate(sent):
        if tag not in tag2idx:
            continue

        ti = tag2idx[tag]
        tag_totals[ti] += 1

        # Emission: which word did this tag produce?
        wi = word2idx.get(word, word2idx['<UNK>'])
        emission_counts[ti][wi] += 1

        # Transition: what tag came before this one?
        if idx > 0:
            prev_tag = sent[idx - 1][1]
            if prev_tag in tag2idx:
                tj = tag2idx[prev_tag]
                transition_counts[tj][ti] += 1

print('✅ Counting complete')
print(f'   Total tag occurrences counted : {int(tag_totals.sum()):,}')
print(f'   Transition matrix shape       : {transition_counts.shape}  (tags × tags)')
print(f'   Emission matrix shape         : {emission_counts.shape}  (tags × vocab)')

✅ Counting complete
   Total tag occurrences counted : 81,021
   Transition matrix shape       : (44, 44)  (tags × tags)
   Emission matrix shape         : (44, 10109)  (tags × vocab)


---
## Cell 3 — Laplace Smoothing

> 🧠 **The zero-probability problem.**  
> Some tag pairs (e.g. Determiner → Determiner) never appear in training.  
> Their count is 0 → probability is 0 → the Viterbi algorithm can never pick any path  
> that passes through them. But in a real test sentence, it might happen!
>
> **Laplace (Add-1) Smoothing** fixes this: pretend we saw every possible  
> combination at least once. Add 1 to every count before dividing.
>
> Formula:  
> `P(tag_j | tag_i) = (count(i→j) + 1) / (count(i) + T)`

In [ ]:
# ── Transition Matrix A ──
# Add 1 to every cell (Laplace), then normalise each row to sum to 1
A = transition_counts + 1.0
A = A / A.sum(axis=1, keepdims=True)   # each row sums to 1

# ── Emission Matrix B ──
# Add 1 to every cell, normalise each row
B = emission_counts + 1.0
B = B / B.sum(axis=1, keepdims=True)   # each row sums to 1

# ── Initial Probability Vector π ──
# Add 1 to every tag, normalise to sum to 1
pi = init_counts + 1.0
pi = pi / pi.sum()

print('✅ Laplace smoothing applied & matrices normalised')
print()
print('Verification — each row of A should sum to 1.0:')
row_sums = A.sum(axis=1)
print(f'   Min row sum: {row_sums.min():.6f}  |  Max row sum: {row_sums.max():.6f}')
print()
print('Verification — π should sum to 1.0:')
print(f'   π sum: {pi.sum():.6f}')

✅ Laplace smoothing applied & matrices normalised

Verification — each row of A should sum to 1.0:
   Min row sum: 1.000000  |  Max row sum: 1.000000

Verification — π should sum to 1.0:
   π sum: 1.000000


---
## Cell 4 — Convert to Log Probabilities

> 🧠 **The underflow problem.**  
> The Viterbi algorithm multiplies many probabilities together, e.g.:  
> `0.62 × 0.55 × 0.01 × 0.23 × ... × 0.08`  
> After ~20 multiplications, this number becomes so tiny that a computer  
> rounds it to **0.0** — called *floating-point underflow*. The calculation breaks.
>
> The fix: take the **log** of every probability.  
> Multiplication becomes addition: `log(a × b) = log(a) + log(b)`  
> Addition never underflows. Problem solved.

In [ ]:
# Convert to log probabilities for numerical stability
# np.log is safe here because Laplace smoothing ensures no zeros
log_A  = np.log(A)
log_B  = np.log(B)
log_pi = np.log(pi)

print('✅ Log probabilities computed')
print()
print('Example: log_A[DT → NN]  (Determiner → Noun transition)')
dt_idx = tag2idx.get('DT', None)
nn_idx = tag2idx.get('NN', None)
if dt_idx is not None and nn_idx is not None:
    print(f'   Raw probability : {A[dt_idx][nn_idx]:.4f}')
    print(f'   Log probability : {log_A[dt_idx][nn_idx]:.4f}')
print()
print('Example: log_B[VBZ → "runs"]  (Verb emitting the word "runs")')
vbz_idx  = tag2idx.get('VBZ', None)
runs_idx = word2idx.get('runs', None)
if vbz_idx is not None and runs_idx is not None:
    print(f'   Raw probability : {B[vbz_idx][runs_idx]:.6f}')
    print(f'   Log probability : {log_B[vbz_idx][runs_idx]:.4f}')

✅ Log probabilities computed

Example: log_A[DT → NN]  (Determiner → Noun transition)
   Raw probability : 0.4706
   Log probability : -0.7538

Example: log_B[VBZ → "runs"]  (Verb emitting the word "runs")
   Raw probability : 0.000845
   Log probability : -7.0765


---
## Cell 5 — Inspect the Transition Matrix

Let's read the matrix like a human — what tag sequences does the model consider likely?

In [ ]:
print('🔍 Most likely tags to FOLLOW each of these common tags:')
print(f'   {"FROM TAG":<8}  →  Top 4 next tags (with probability)')
print('   ' + '-' * 55)

inspect_tags = ['DT', 'NN', 'VBD', 'JJ', 'IN', 'PRP', 'MD']

for from_tag in inspect_tags:
    if from_tag not in tag2idx:
        continue
    i = tag2idx[from_tag]
    top4_idx = np.argsort(A[i])[::-1][:4]
    top4 = [(idx2tag[j], A[i][j]) for j in top4_idx]
    top4_str = '  '.join(f'{t}({p:.2f})' for t, p in top4)
    print(f'   {from_tag:<8}  →  {top4_str}')

print()
print('✅ These probabilities match English grammar intuition!')
print('   DT → NN makes sense: "the dog", "a cat", "the city"')
print('   MD → VB makes sense: "can run", "will go", "should eat"')

🔍 Most likely tags to FOLLOW each of these common tags:
   FROM TAG  →  Top 4 next tags (with probability)
   -------------------------------------------------------
   DT        →  NN(0.47)  JJ(0.20)  NNP(0.13)  NNS(0.08)
   NN        →  IN(0.24)  NN(0.13)  ,(0.11)  .(0.10)
   VBD       →  (0.28)  DT(0.13)  IN(0.11)  VBN(0.09)
   JJ        →  NN(0.44)  NNS(0.24)  JJ(0.06)  IN(0.06)
   IN        →  DT(0.32)  NNP(0.15)  NN(0.11)  JJ(0.10)
   PRP       →  VBD(0.24)  VBZ(0.19)  VBP(0.17)  MD(0.13)
   MD        →  VB(0.77)  RB(0.16)  (0.01)  ``(0.01)

✅ These probabilities match English grammar intuition!
   DT → NN makes sense: "the dog", "a cat", "the city"
   MD → VB makes sense: "can run", "will go", "should eat"


---
## Cell 6 — Inspect the Emission Matrix

What words does each tag most commonly produce?

In [ ]:
# Reverse vocab for lookup: index → word
idx2word = {i: w for w, i in word2idx.items()}

print('🔍 Most likely words emitted by each tag:')
print(f'   {"TAG":<8}  →  Top 6 words')
print('   ' + '-' * 60)

inspect_tags = ['DT', 'IN', 'NN', 'VBD', 'JJ', 'RB', 'PRP', 'MD']

for tag in inspect_tags:
    if tag not in tag2idx:
        continue
    ti = tag2idx[tag]
    # Get top 6 — skip <UNK>
    top_idx = np.argsort(B[ti])[::-1]
    top_words = []
    for wi in top_idx:
        w = idx2word[wi]
        if w != '<UNK>':
            top_words.append(w)
        if len(top_words) == 6:
            break
    print(f'   {tag:<8}  →  {top_words}')

print()
print('✅ Emission matrix looks linguistically sensible!')

🔍 Most likely words emitted by each tag:
   TAG       →  Top 6 words
   ------------------------------------------------------------
   DT        →  ['the', 'a', 'an', 'this', 'some', 'that']
   IN        →  ['of', 'in', 'for', 'on', 'that', 'at']
   NN        →  ['%', 'company', 'year', 'market', 'trading', 'program']
   VBD       →  ['said', 'was', 'were', 'had', 'did', 'rose']
   JJ        →  ['new', 'other', 'last', 'many', 'such', 'japanese']
   RB        →  ["n't", 'also', 'not', 'even', 'only', 'so']
   PRP       →  ['it', 'he', 'they', 'i', 'she', 'we']
   MD        →  ['will', 'would', 'could', 'can', 'may', 'might']

✅ Emission matrix looks linguistically sensible!


---
## Cell 7 — Save the HMM

Save all three matrices so Step 03 (Viterbi) can load them directly.

In [ ]:
hmm = {
    # Raw probabilities (kept for inspection)
    'A'       : A,
    'B'       : B,
    'pi'      : pi,
    # Log probabilities (used during Viterbi decoding)
    'log_A'   : log_A,
    'log_B'   : log_B,
    'log_pi'  : log_pi,
    # Lookups
    'tag2idx' : tag2idx,
    'idx2tag' : idx2tag,
    'word2idx': word2idx,
    'idx2word': idx2word,
    'tagset'  : tagset,
    'vocab'   : vocab,
}

with open('hmm_model.pkl', 'wb') as f:
    pickle.dump(hmm, f)

print('✅ HMM saved → hmm_model.pkl')
print()
print('📦 What was saved:')
print(f'   A        : Transition matrix      {A.shape}  (tags × tags)')
print(f'   B        : Emission matrix        {B.shape}  (tags × vocab)')
print(f'   pi       : Initial probabilities  ({len(pi)},)')
print(f'   log_A/B/pi: Log versions for Viterbi')
print()
print('🚀 Ready for Step 03 — Viterbi Decoding')

✅ HMM saved → hmm_model.pkl

📦 What was saved:
   A        : Transition matrix      (44, 44)  (tags × tags)
   B        : Emission matrix        (44, 10109)  (tags × vocab)
   pi       : Initial probabilities  (44,)
   log_A/B/pi: Log versions for Viterbi

🚀 Ready for Step 03 — Viterbi Decoding


---
## ✅ Step 02 Complete — Summary

| What we built | Shape | What it captures |
|---|---|---|
| Transition Matrix **A** | (T × T) | Grammar patterns between tags |
| Emission Matrix **B** | (T × V) | Which words belong to which tags |
| Initial Vector **π** | (T,) | How sentences tend to start |

All three are smoothed (no zeros) and stored as both raw and log probabilities.

**Next up → Step 03: Viterbi Decoding — using the HMM to tag new sentences**